# Data Leakage Prevention (DLP) ML Classifier - Training Pipeline

This Google Colab notebook fine-tunes a **Mistral 7B** base model using **Unsloth** and **LoRA**.
The dataset should be located in your Google Drive (in `.jsonl` format). The objective is to teach the model to classify risk based on formatted JSON features and text into exactly one of `ALLOW`, `REDACT`, `ESCALATE`, or `BLOCK`.

**Instructions**
1. Open this notebook in Google Colab (`File -> Upload notebook`).
2. Ensure you have selected a **T4 GPU** runtime (`Runtime -> Change runtime type -> Hardware accelerator: T4 GPU`).
3. Run all cells sequentially.

In [ ]:
# --- Section 1: Setup ---
# Install Unsloth and required dependencies for Colab
# Unsloth is optimized for single T4/A100 GPUs and significantly speeds up fine-tuning while reducing VRAM usage.

%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
# --- Section 2: Mount Google Drive ---
from google.colab import drive

# Mount Google Drive manually
drive.mount('/content/drive')

# Configuration Variables - Adjust these if needed
DATASET_PATH = "/content/drive/MyDrive/dlp_dataset.jsonl"
OUTPUT_MODEL_PATH = "/content/drive/MyDrive/dlp_mistral_lora_outputs"

In [ ]:
# --- Section 3: Load Dataset ---
import json
from datasets import load_dataset

print(f"Loading dataset from: {DATASET_PATH}")
# Read directly as a huggingface Dataset
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

print(f"Loaded {len(dataset)} training samples.")
# Display the first sample to verify structure
for key, value in dataset[0].items():
    print(f"[{key}]:", value)

In [ ]:
# --- Section 4: Model & Tokenizer Loading ---
import torch
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

# Model Configuration
max_seq_length = 1024 # Standard context size for short text classification
dtype = None # Auto-detect. Use Float16 on T4 V100, Bfloat16 on Ampere+
load_in_4bit = True # 4-bit quantization via bitsandbytes to fit in Colab limits

print("Optimizing Mistral 7B via Unsloth & BitsandBytes (4-bit)...")
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit", # Best small Mistral 7B variant fitting T4 seamlessly
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Apply standard Mistral chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "mistral", 
)

In [ ]:
# --- Section 5: Preprocessing ---
# This is our unified prompt logic explicitly instructing the model on what to output.
# We present the input text along with pre-extracted features and surface components.

def format_prompts(examples):
    """
    Batched mapping function to convert dataset rows into the exact prompt string.
    """
    surfaces = examples["surface"]
    features_list = examples["features"]
    texts = examples["text"]
    labels = examples["label"]
    
    formatted_texts = []
    
    for surface, features, text, label in zip(surfaces, features_list, texts, labels):
        # Convert dictionary to newline-separated properties
        features_str = "\n".join([f"{k}={v}" for k, v in features.items()])
        
        prompt = f"""You are a classifier for AI agents Data Leakage Prevention (DLP).

Your task is to analyze the input and classify its risk level into EXACTLY one of:
ALLOW, REDACT, ESCALATE, BLOCK.

You are given:
- SURFACE: where the data appears (OUTPUT:LLM's final output to the user, TOOL_ARGS: arguments for the tool calling, TOOL_RESULT: result from the tool)
- FEATURES: extracted signals from deterministic analysis
- TEXT: the raw content

You must follow these rules:

DEFINITIONS:

- ALLOW:
  Content is safe. It may include examples, placeholders, documentation, or synthetic data.
  No real sensitive data is exposed.

- REDACT:
  Contains limited personal data (e.g., one or a few emails, phone numbers, or credit cards).
  Data is real or realistic but not highly sensitive.

- ESCALATE:
  Content is ambiguous or potentially risky.
  Examples:
    - multiple PII entries
    - high-entropy strings that may be tokens
    - partial or unclear secrets
    - mixed benign and sensitive signals
  If you are unsure, choose ESCALATE.

- BLOCK:
  Content clearly contains highly sensitive data or secrets.
  Examples:
    - API keys, tokens, credentials
    - database connection strings with passwords
    - private keys
    - authentication headers
  Always BLOCK if a real secret is present.

FEATURE USAGE GUIDELINES:

- num_emails > 0 -> indicates presence of PII (REDACT or ESCALATE depending on amount)
- num_phones > 0 -> PII signal (REDACT or ESCALATE)
- num_credit_cards > 0 -> sensitive PII (REDACT or ESCALATE)
- has_valid_credit_card = true -> strong sensitive signal (prefer REDACT or ESCALATE)
- num_secrets_detected > 0 -> strong signal for BLOCK
- has_api_key_pattern = true -> strong signal for BLOCK
- has_db_connection = true -> strong signal for BLOCK (especially if credentials present)
- has_private_key = true -> definite BLOCK
- high max_entropy or avg_entropy -> may indicate tokens/secrets -> ESCALATE or BLOCK depending on context
- is_example_context = true -> reduces risk (may allow ALLOW)
- is_code_context = true -> may indicate examples or documentation (reduce risk unless real secrets present)
- multiple risk signals combined -> increase severity

PRIORITY RULES:

1. If real secrets or credentials are present -> BLOCK
2. Else if clear PII is present -> REDACT
3. Else if ambiguous or suspicious -> ESCALATE
4. Else -> ALLOW

IMPORTANT:

- Use TEXT as the primary source of truth
- Use FEATURES as supporting signals, not absolute truth
- If conflicting signals exist, prioritize safety
- If unsure, choose ESCALATE
- Do NOT explain your answer
- Output ONLY one word: ALLOW, REDACT, ESCALATE, or BLOCK

### Input:
[SURFACE={surface}]

[FEATURES]
{features_str}

[TEXT]
{text}"""
        
        # Build conversational turn for Mistral Chat template
        convo = [
            {"role": "user", "content": prompt},
            {"role": "model", "content": label}
        ]
        
        # Apply the chat template defined from the tokenizer
        # Remove bos_token to avoid double additions during SFT training
        formatted = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        if formatted.startswith(tokenizer.bos_token):
            formatted = formatted.replace(tokenizer.bos_token, "", 1)
            
        formatted_texts.append(formatted)
        
    return {"formatted_text": formatted_texts}

# Map and format the dataset instances right before training
print("Formatting the dataset instances...")
dataset = dataset.map(format_prompts, batched=True)
print("Sample Prompt Input:")
print("===============")
print(dataset[0]["formatted_text"])
print("===============")

In [ ]:
# --- Section 6: Apply LoRA Adapters ---

# Instead of updating billions of parameters, LoRA only updates ~1% 
# For Mistral 7B, Unsloth allows targeted multimodal tuning if needed, but we keep vision layers off.
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text classification!
    finetune_language_layers   = True,  
    finetune_attention_modules = True,  
    finetune_mlp_modules       = True,  
    r = 16, # Rank mapping size (16 is a solid default)
    lora_alpha = 16,
    lora_dropout = 0, # Optimized: Keep at 0
    bias = "none",    # Optimized: Keep at "none"
    use_gradient_checkpointing = "unsloth", # Crucial for fitting into T4's 15GB limits
    random_state = 3407,
)

print(f"PEFT / LoRA modules injected!")
model.print_trainable_parameters()

In [ ]:
# --- Section 7: Training Setup ---
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only
import torch

# Initialize HF standard Trainer via PEFT/Unsloth integrations
# Configure hyper-params tailored for T4 VRAM.
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,              # Full 6000 sample dataset
    args = SFTConfig(
        dataset_text_field = "formatted_text",# Point to our dynamically built strings
        max_seq_length = max_seq_length,
        # packing = False,                # Disabling sequence packing allows precise label handling 
        dataset_num_proc = 2,             # Colab limits parallel workers
        per_device_train_batch_size = 1,  # Set batch size tight to avoid Mistral OOM on 10+ GB models
        gradient_accumulation_steps = 8,  # Effective Batch Size = 1 * 8 = 8
        warmup_steps = 5,
        num_train_epochs = 2,             # 2 Epochs = 12000 total iterations, fast and reliable for text-class
        learning_rate = 2e-4,             # Industry standard learning rate for Models w/ LoRA
        fp16 = not FastModel.is_bfloat16_supported(), # Automatically uses fp16 on T4 GPUs
        bf16 = FastModel.is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",             # Using 8-bit optimizer cuts VRAM overhead heavily
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",           # Temp storage during training
        report_to = "none",               # Prevent wandb errors in colab
    ),
)

# Apply response-only training to ignore user prompt from loss calculation
# This improves instruction following.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "[INST]",
    response_part = "[/INST]",
)

# Benchmark memory footprint before training:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name} | Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved currently.")

In [ ]:
# --- Section 8: Execute Training Process ---
print("Initiating full training loop over 6000 samples...")
trainer_stats = trainer.train()

print(f"\nFinal Training loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# --- Section 9: Save Weights to Google Drive ---
print(f"Persisting model adapters and tokenizer weights directly to Google Drive...")
print(f"Path: {OUTPUT_MODEL_PATH}")

model.save_pretrained(OUTPUT_MODEL_PATH)      # Saves adapter_config.json and adapter_model.bin/safetensors
tokenizer.save_pretrained(OUTPUT_MODEL_PATH)  # Saves tokenizer_config.json

print("Save completed successfully. You can verify it within your Google Drive.")

In [ ]:
# --- Section 10: Export and Download zip ---
import shutil
from google.colab import files

print("Zipping the LoRA adapter folder locally for direct download...")
ARCHIVE_NAME = "/content/dlp_model_lora"

# Creates /content/dlp_model_lora.zip
shutil.make_archive(ARCHIVE_NAME, 'zip', OUTPUT_MODEL_PATH)

print(f"Triggering direct download of {ARCHIVE_NAME}.zip. This might take a few moments based on browser constraints...")
# Automatically pops up a browser file download dialog
files.download(f"{ARCHIVE_NAME}.zip")
print("Finished!")